# Rules Engine 2.1 · Enhancement Demo

## Decisions you can author, trust, and explain

A live, mixed-audience walkthrough of the new authoring contract, strict type safety, chained decisions, explicit assignment outcomes, and coverage diagnostics.

### Presenter run-of-show

This notebook is designed for a **15–20 minute demo**. Use **Run All** before the meeting; the core path is intentionally Java-free.

1. **2 minutes — Why it matters:** contract-driven authoring, safer changes, explainable outcomes.
2. **8 minutes — Live decision:** compile, validate, evaluate, and inspect one account.
3. **5 minutes — Engineering proof:** fail-fast types, manifest details, and rule coverage.
4. **Optional — Databricks appendix:** public Spark API, full audit, applied business rows, and production coverage.


In [ ]:
from __future__ import annotations

from collections import Counter
from hashlib import sha256
from html import escape
import json
from pathlib import Path
import sys

# Find a source checkout when the package is not already installed.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'rules_engine').is_dir()),
    None,
)
if PROJECT_ROOT is not None and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from rules_engine import (
    FunctionRegistry,
    RulesetValidator,
    YamlRulesetCompiler,
    __version__,
    build_authoring_manifest,
    register_standard_functions,
)
from rules_engine.exceptions import CompilationError
from rules_engine.human_readable import HumanReadableRulesetFormatter
from rules_engine.models import LiteralOperand
from rules_engine.runtime import SparkRowEvaluator

DATABRICKS_DISPLAY_HTML = globals().get('displayHTML')
NATIVE_DISPLAY = globals().get('display')

try:
    from IPython.display import HTML as IPythonHTML
    from IPython.display import display as ipython_display
except ImportError:  # Lets the notebook source be verified without Jupyter installed.
    class IPythonHTML(str):
        pass
    def ipython_display(value):
        print(str(value)[:240])

STYLES = '''
<style>
.re-wrap{font-family:Inter,Segoe UI,Arial,sans-serif;color:#17232b;margin:10px 0 22px}
.re-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px}
.re-card{background:#fff;border:1px solid #dce6e8;border-radius:14px;padding:16px;box-shadow:0 5px 16px rgba(13,52,62,.06)}
.re-metric{font-size:30px;font-weight:760;color:#0f6b66;line-height:1.05}.re-label{font-size:12px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#60727b;margin-top:7px}
.re-note{font-size:13px;color:#60727b;margin-top:5px}.re-ok{color:#087a58}.re-warn{color:#a85c00}.re-bad{color:#b42318}
.re-pill{display:inline-block;padding:4px 9px;border-radius:999px;background:#e7f5f2;color:#0a6860;font-size:11px;font-weight:750;margin:2px 4px 2px 0}
.re-pill.gray{background:#edf1f2;color:#596970}.re-pill.amber{background:#fff0d6;color:#8a5200}.re-pill.red{background:#fee9e7;color:#9d251d}
.re-table{width:100%;border-collapse:separate;border-spacing:0;background:#fff;border:1px solid #dce6e8;border-radius:14px;overflow:hidden;font-size:13px}
.re-table th{background:#eff7f6;color:#31515a;text-align:left;padding:10px 12px;font-size:11px;text-transform:uppercase;letter-spacing:.06em}
.re-table td{padding:11px 12px;border-top:1px solid #e7edef;vertical-align:top}.re-table tr:hover td{background:#fbfdfd}
.re-flow{display:flex;gap:8px;align-items:stretch;flex-wrap:wrap}.re-stage{flex:1;min-width:135px;background:#fff;border:1px solid #cfe0e1;border-top:4px solid #18a999;border-radius:12px;padding:13px}.re-stage b{display:block;color:#123b44;margin-bottom:6px}.re-arrow{align-self:center;color:#7aa2a4;font-size:22px}
.re-code{font-family:ui-monospace,SFMono-Regular,Consolas,monospace;background:#081f2c;color:#d9f1ee;border-radius:12px;padding:14px;white-space:pre-wrap;font-size:12px;line-height:1.5}
.re-callout{border-left:5px solid #18a999;background:#effaf8;border-radius:10px;padding:14px 16px}.re-error{border-left-color:#df3f35;background:#fff2f0}
.re-timeline{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.re-event{background:#fff;border:1px solid #d7e3e5;border-radius:12px;padding:12px 14px;min-width:145px}.re-event strong{color:#0f6b66}.re-connector{color:#79a1a4;font-size:20px}
</style>
'''
def render_html(*blocks):
    """Render one self-contained HTML document in Databricks or Jupyter."""
    document = STYLES + ''.join(blocks)
    if callable(DATABRICKS_DISPLAY_HTML):
        DATABRICKS_DISPLAY_HTML(document)
    else:
        ipython_display(IPythonHTML(document))

def display_value(value):
    """Use Databricks' native table display when present, else IPython."""
    if callable(NATIVE_DISPLAY):
        NATIVE_DISPLAY(value)
    else:
        ipython_display(value)

def show_metrics(items):
    cards = ''.join(
        f"<div class='re-card'><div class='re-metric'>{escape(str(value))}</div>"
        f"<div class='re-label'>{escape(label)}</div><div class='re-note'>{escape(note)}</div></div>"
        for value, label, note in items
    )
    return f"<div class='re-wrap re-grid'>{cards}</div>"

def show_table(rows, columns):
    header = ''.join(f'<th>{escape(label)}</th>' for _, label in columns)
    body = ''.join(
        '<tr>' + ''.join(f"<td>{row.get(key, '')}</td>" for key, _ in columns) + '</tr>'
        for row in rows
    )
    return f"<div class='re-wrap'><table class='re-table'><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"

def show_bars(items, *, width=760):
    max_value = max((value for _, value, _ in items), default=1) or 1
    row_height, label_width, bar_width = 42, 190, width - 300
    svg_rows = []
    for index, (label, value, flag) in enumerate(items):
        y = 20 + index * row_height
        fill = '#df3f35' if flag == 'dead' else ('#e89a2f' if flag == 'broad' else '#18a999')
        actual = max(2, int(bar_width * value / max_value)) if value else 2
        svg_rows.append(
            f"<text x='0' y='{y + 14}' font-size='12' fill='#263a42'>{escape(label)}</text>"
            f"<rect x='{label_width}' y='{y}' width='{bar_width}' height='20' rx='10' fill='#e9eff0'/>"
            f"<rect x='{label_width}' y='{y}' width='{actual}' height='20' rx='10' fill='{fill}'/>"
            f"<text x='{label_width + bar_width + 12}' y='{y + 14}' font-size='12' font-weight='700' fill='#263a42'>{value:.0%}</text>"
        )
    height = 24 + len(items) * row_height
    return f"<div class='re-wrap re-card'><svg viewBox='0 0 {width} {height}' width='100%' role='img'>{''.join(svg_rows)}</svg></div>"

print(f'Ready — rules_engine {__version__} from {PROJECT_ROOT or "installed package"}')


## 1 · Executive view — what changed?

The enhancement is more than additional operators. It creates a shared contract from rule authoring through production evidence:

- **Faster, safer authoring:** tools discover valid choices from the installed engine.
- **Earlier defect detection:** known literal types are normalized and rejected before data processing.
- **Traceable decisions:** every result carries matched rule IDs and explicit assignment state.
- **Operational confidence:** coverage exposes no-match, dead, and suspiciously broad rules.


In [ ]:
registry = register_standard_functions(FunctionRegistry())
manifest = build_authoring_manifest(registry)
manifest_json = json.dumps(manifest, sort_keys=True, separators=(',', ':'))
manifest_fingerprint = sha256(manifest_json.encode()).hexdigest()[:12]

render_html(show_metrics([
    (len(manifest['comparison_operators']), 'Comparison operators', 'One canonical vocabulary'),
    (len(manifest['functions']), 'Standard functions', 'Typed, permissioned contracts'),
    (len(manifest['literal_type_hints']), 'Canonical literal types', 'Aliases remain accepted'),
    (manifest_fingerprint, 'Manifest fingerprint', 'Deterministic and JSON-safe'),
]))


In [ ]:
stages = [
    ('1 · Author', 'YAML or UI choices come from one manifest'),
    ('2 · Compile', 'Reject ambiguous shape and unsafe literals'),
    ('3 · Validate', 'Check semantics, dependencies, and Spark types'),
    ('4 · Evaluate', 'Ordered, atomic row decisions'),
    ('5 · Explain', 'Keyed results, audit history, and coverage'),
]
parts = []
for index, (title, note) in enumerate(stages):
    parts.append(f"<div class='re-stage'><b>{escape(title)}</b><span>{escape(note)}</span></div>")
    if index < len(stages) - 1:
        parts.append("<div class='re-arrow'>→</div>")
render_html(f"<div class='re-wrap re-flow'>{''.join(parts)}</div>")


## 2 · Live scenario — account review triage

We will route six accounts using four readable rules. The scenario intentionally demonstrates multiple matches, a later rule reading an earlier assignment, an explicit null clear, a clean no-match, and a dead rule.


In [ ]:
ruleset_yaml = '''
ruleset_id: account_review_demo
ruleset_name: Account Review Demo
version: "2.1-demo"
description: Mixed-audience demonstration of the 2.1 contract
owner: Rules Product Team
owner_department: Engineering
rules:
  - rule_id: material_open
    rule_name: Material open exposure
    rule_order: 10
    when:
      condition_group_id: material_open_all
      all:
        - condition_id: account_is_open
          left: {field: status}
          operator: eq
          right: {literal: OPEN, value_type: string}
        - condition_id: exposure_is_material
          left: {field: exposure, default_if_null: 0}
          operator: ge
          right: {literal: 100000, value_type: integer}
    assign:
      - assignment_id: set_high_risk
        target_field: risk_band
        value: {literal: HIGH, value_type: string}
      - assignment_id: route_enhanced_review
        target_field: route
        value: {literal: Enhanced review, value_type: string}
      - assignment_id: set_two_day_sla
        target_field: sla_days
        value: {literal: 2, value_type: integer}

  - rule_id: sensitive_region_escalation
    rule_name: Escalate a high-risk sensitive region
    rule_order: 20
    when:
      condition_group_id: sensitive_region_all
      all:
        - condition_id: prior_rule_set_high
          left: {assigned: risk_band}
          operator: eq
          right: {literal: HIGH, value_type: string}
        - condition_id: region_is_sensitive
          left: {field: region}
          operator: in
          right: {literal: [HEIGHTENED, RESTRICTED], value_type: string}
    assign:
      - assignment_id: set_critical_risk
        target_field: risk_band
        value: {literal: CRITICAL, value_type: string}
      - assignment_id: route_executive_review
        target_field: route
        value: {literal: Executive review, value_type: string}
      - assignment_id: set_one_day_sla
        target_field: sla_days
        value: {literal: 1, value_type: integer}

  - rule_id: clear_closed_hold
    rule_name: Clear holds on closed accounts
    rule_order: 30
    when:
      all:
        - condition_id: account_is_closed
          left: {field: status}
          operator: eq
          right: {literal: CLOSED, value_type: string}
    assign:
      - assignment_id: clear_hold_reason
        target_field: hold_reason
        value: {literal: null, value_type: string}

  - rule_id: archive_extreme_closed
    rule_name: Archive an extreme closed exposure
    rule_order: 40
    when:
      all:
        - left: {field: status}
          operator: eq
          right: {literal: CLOSED, value_type: string}
        - left: {field: exposure}
          operator: ge
          right: {literal: 1000000, value_type: integer}
    assign:
      archive_review: {literal: true, value_type: boolean}
'''

compiler = YamlRulesetCompiler()
ruleset = compiler.compile_text(ruleset_yaml)
validation = RulesetValidator(registry).validate(ruleset)
assert validation.passed, validation.to_text()

render_html(
    f"<div class='re-wrap re-callout'><strong class='re-ok'>✓ Compile + semantic validation passed</strong>"
    f"<div class='re-note'>{len(ruleset.rules)} immutable compiled rules · owner={escape(ruleset.owner or '')} · version={escape(ruleset.version)}</div></div>"
)


In [ ]:
formatter = HumanReadableRulesetFormatter()
descriptions = {row['rule_id']: row for row in formatter.describe_rules(ruleset)}
rule_cards = []
for rule in sorted(ruleset.rules, key=lambda item: item.rule_order):
    description = descriptions[rule.rule_id]
    rule_cards.append(
        f"<div class='re-card'><span class='re-pill'>order {rule.rule_order}</span>"
        f"<h4 style='margin:9px 0 7px'>{escape(rule.rule_name)}</h4>"
        f"<div class='re-note'>{escape(description['rule_logic'])}</div>"
        f"<div style='margin-top:10px'><b>Then:</b> {escape(description['match_payload'])}</div></div>"
    )
render_html(f"<div class='re-wrap re-grid'>{''.join(rule_cards)}</div>")


## 3 · One input, two useful outputs

For a portable demo, the next cell calls the same row-semantic evaluator used inside Spark workers. In production, the public DataFrame API returns a keyed evidence view separately from the business rows with final assignments applied.


In [ ]:
source_rows = [
    {'account_id': 'A-100', 'status': 'OPEN',    'exposure': 250000, 'region': 'STANDARD',   'risk_band': 'LOW', 'route': None, 'sla_days': None, 'hold_reason': None},
    {'account_id': 'A-200', 'status': 'OPEN',    'exposure': 500000, 'region': 'RESTRICTED', 'risk_band': 'LOW', 'route': None, 'sla_days': None, 'hold_reason': 'screening'},
    {'account_id': 'A-300', 'status': 'CLOSED',  'exposure': 5000,   'region': 'STANDARD',   'risk_band': 'LOW', 'route': None, 'sla_days': None, 'hold_reason': 'investigation'},
    {'account_id': 'A-400', 'status': 'OPEN',    'exposure': 50000,  'region': 'HEIGHTENED', 'risk_band': 'LOW', 'route': None, 'sla_days': None, 'hold_reason': 'manual'},
    {'account_id': 'A-500', 'status': 'OPEN',    'exposure': 150000, 'region': 'HEIGHTENED', 'risk_band': 'LOW', 'route': None, 'sla_days': None, 'hold_reason': None},
    {'account_id': 'A-600', 'status': 'PENDING', 'exposure': 1000,   'region': 'STANDARD',   'risk_band': 'LOW', 'route': None, 'sla_days': None, 'hold_reason': None},
]

row_evaluator = SparkRowEvaluator.without_repository(registry)
evaluated_rows = []
for source in source_rows:
    evidence = row_evaluator.evaluate_row(ruleset, source)
    applied = dict(source)
    actions = []
    for target, outcome in evidence['assign'].items():
        if outcome['applied']:
            before = applied.get(target)
            applied[target] = outcome['value']
            after = '∅' if outcome['value'] is None else outcome['value']
            actions.append(f'{target}: {before!r} → {after}')
    evaluated_rows.append({'source': source, 'evidence': evidence, 'applied': applied, 'actions': actions})

table_rows = []
for item in evaluated_rows:
    source, evidence, applied = item['source'], item['evidence'], item['applied']
    match_html = ''.join(f"<span class='re-pill'>{escape(rule_id)}</span>" for rule_id in evidence['matched_rule_ids']) or "<span class='re-pill gray'>clean no-match</span>"
    final_html = (
        f"<b>{escape(str(applied.get('risk_band')))}</b><br>"
        f"<span class='re-note'>{escape(str(applied.get('route') or 'no route'))} · SLA {escape(str(applied.get('sla_days') or '—'))}</span>"
    )
    action_html = '<br>'.join(escape(action) for action in item['actions']) or "<span class='re-note'>Input retained</span>"
    table_rows.append({
        'account': f"<b>{escape(source['account_id'])}</b>",
        'signals': f"{escape(source['status'])} · ${source['exposure']:,}<br><span class='re-note'>{escape(source['region'])}</span>",
        'matched': match_html,
        'final': final_html,
        'actions': action_html,
    })
render_html(show_table(table_rows, [('account', 'Account'), ('signals', 'Input signals'), ('matched', 'Matched rules'), ('final', 'Final decision'), ('actions', 'Assignment action')]))

by_id = {item['source']['account_id']: item for item in evaluated_rows}
assert by_id['A-200']['evidence']['matched_rule_ids'] == ['material_open', 'sensitive_region_escalation']
assert by_id['A-200']['applied']['risk_band'] == 'CRITICAL'
assert by_id['A-300']['evidence']['assign']['hold_reason'] == {'applied': True, 'value': None}
assert by_id['A-400']['evidence']['assign']['hold_reason'] == {'applied': False, 'value': None}
assert by_id['A-400']['applied']['hold_reason'] == 'manual'


In [ ]:
def outcome_label(item):
    applied = item['applied']
    if applied.get('route') == 'Executive review':
        return 'Executive review'
    if applied.get('route') == 'Enhanced review':
        return 'Enhanced review'
    if 'clear_closed_hold' in item['evidence']['matched_rule_ids']:
        return 'Closed hold cleared'
    return 'No decision change'

outcomes = Counter(outcome_label(item) for item in evaluated_rows)
total = len(evaluated_rows)
render_html(
    show_metrics([
    (outcomes['Executive review'], 'Executive reviews', 'Highest-priority route'),
    (outcomes['Enhanced review'], 'Enhanced reviews', 'Material open exposure'),
    (outcomes['Closed hold cleared'], 'Explicit clears', 'Applied null, not no-op'),
    (outcomes['No decision change'], 'No change', 'Original values preserved'),
    ]),
    show_bars([(label, count / total, '') for label, count in outcomes.most_common()]),
)


## 4 · Explain one decision — A-200

A-200 demonstrates ordered rule composition. The first rule establishes a reusable decision; the second rule reads that committed value through `assigned` and escalates it. The final value wins, while production full audit retains both assignment events and their provenance.


In [ ]:
selected = by_id['A-200']
state = dict(selected['source'])
events = ["<div class='re-event'><strong>Input</strong><br>risk_band = LOW</div>"]
for rule_id in selected['evidence']['matched_rule_ids']:
    rule = next(rule for rule in ruleset.rules if rule.rule_id == rule_id)
    changes = []
    for assignment in rule.assignments:
        assert isinstance(assignment.value, LiteralOperand)
        old_value = state.get(assignment.target_field)
        state[assignment.target_field] = assignment.value.value
        changes.append(f"{assignment.target_field}: {old_value!r} → {assignment.value.value!r}")
    logic = descriptions[rule_id]['rule_logic']
    events.extend([
        "<div class='re-connector'>→</div>",
        f"<div class='re-event'><strong>{escape(rule.rule_name)}</strong><br>"
        f"<span class='re-note'>{escape(logic)}</span><br>{'<br>'.join(escape(change) for change in changes)}</div>",
    ])
events.extend([
    "<div class='re-connector'>→</div>",
    f"<div class='re-event'><strong>Final</strong><br>risk_band = {escape(str(state['risk_band']))}<br>route = {escape(str(state['route']))}</div>",
])
render_html(f"<div class='re-wrap re-timeline'>{''.join(events)}</div>")


## 5 · Engineering proof — bad types fail at the source

The strengthened authoring contract no longer lets a fractional value masquerade as an integer and drift into schema validation or runtime coercion. Known scalar hints normalize to their declared runtime type; incompatible values stop compilation immediately.


In [ ]:
invalid_yaml = '''
ruleset_id: invalid_demo
ruleset_name: Invalid Demo
version: "1"
rules:
  - rule_name: Fractional integer
    when:
      all:
        - left: {field: exposure}
          operator: ge
          right: {literal: 100000.5, value_type: integer}
    assign: {route: review}
'''
try:
    compiler.compile_text(invalid_yaml)
    raise AssertionError('The invalid ruleset unexpectedly compiled.')
except CompilationError as exc:
    error_text = str(exc)

assert 'fractional component' in error_text
render_html(
    "<div class='re-wrap re-callout re-error'><strong class='re-bad'>✕ Compilation stopped before evaluation</strong>"
    f"<div class='re-code' style='margin-top:10px'>value_type: integer\nliteral: 100000.5\n\n{escape(error_text)}</div></div>"
)


## 6 · Engineering proof — authoring tools discover the contract

Editors and APIs no longer need hand-copied lists of operators or functions. The manifest is deterministic, JSON-compatible, versioned, and reflects the functions registered in the current environment. Implementation references are deliberately excluded.


In [ ]:
shape_counts = Counter(item['right_operand_shape'] for item in manifest['comparison_operators'])
operator_rows = []
for shape in ('none', 'any', 'collection', 'pair'):
    names = [item['name'] for item in manifest['comparison_operators'] if item['right_operand_shape'] == shape]
    operator_rows.append({
        'shape': f"<span class='re-pill'>{escape(shape)}</span>",
        'count': str(shape_counts[shape]),
        'operators': escape(', '.join(names)),
    })
to_decimal_contract = next(item for item in manifest['functions'] if item['function_name'] == 'to_decimal')
assert 'implementation_reference' not in to_decimal_contract
arguments = ''.join(
    f"<div><span class='re-pill'>{escape(argument['name'])}</span> type={escape(argument['type_hint'])} · "
    f"{'required' if argument['required'] else 'optional'}"
    f"{(' · allowed=' + escape(str(argument.get('allowed_values')))) if argument.get('allowed_values') else ''}</div>"
    for argument in to_decimal_contract['arguments']
)
render_html(
    show_table(operator_rows, [('shape', 'Right operand shape'), ('count', 'Count'), ('operators', 'Canonical operators')]),
    f"<div class='re-wrap re-card'><div class='re-label'>Example function contract</div>"
    f"<h3 style='margin:8px 0'>to_decimal → {escape(str(to_decimal_contract['return_type_hint']))}</h3>"
    f"<p>{escape(str(to_decimal_contract['description']))}</p>{arguments}"
    f"<div style='margin-top:10px'><span class='re-pill'>condition: {to_decimal_contract['allowed_in_condition_flag']}</span>"
    f"<span class='re-pill'>assignment: {to_decimal_contract['allowed_in_assignment_flag']}</span></div></div>"
)


## 7 · Engineering + operations — is the rule set behaving as intended?

Coverage makes gaps and overreach visible. The production analyzer performs this aggregation with the production Spark evaluator; here we calculate the same presentation metrics from the six already-evaluated demo rows.


In [ ]:
broad_threshold = 0.40
coverage_rows = []
bar_items = []
for rule in sorted(ruleset.rules, key=lambda item: item.rule_order):
    count = sum(rule.rule_id in item['evidence']['matched_rule_ids'] for item in evaluated_rows)
    first_count = sum(
        bool(item['evidence']['matched_rule_ids']) and item['evidence']['matched_rule_ids'][0] == rule.rule_id
        for item in evaluated_rows
    )
    rate = count / total
    flag = 'dead' if count == 0 else ('broad' if rate >= broad_threshold else '')
    status = "<span class='re-pill red'>dead</span>" if flag == 'dead' else ("<span class='re-pill amber'>broad</span>" if flag == 'broad' else "<span class='re-pill'>healthy</span>")
    coverage_rows.append({
        'rule': f"<b>{escape(rule.rule_name)}</b><br><span class='re-note'>{escape(rule.rule_id)}</span>",
        'matches': str(count),
        'first': str(first_count),
        'rate': f'{rate:.0%}',
        'status': status,
    })
    bar_items.append((rule.rule_name, rate, flag))

no_match_count = sum(not item['evidence']['matched'] for item in evaluated_rows)
render_html(
    show_metrics([
    (total, 'Rows evaluated', 'Small representative tape'),
    (no_match_count, 'Clean no-match rows', 'Candidates for rule-gap review'),
    (sum(row['status'].find('dead') >= 0 for row in coverage_rows), 'Dead rules', 'Zero observed matches'),
    (sum(row['status'].find('broad') >= 0 for row in coverage_rows), 'Broad rules', f'At or above {broad_threshold:.0%}'),
    ]),
    show_bars(bar_items),
    show_table(coverage_rows, [('rule', 'Rule'), ('matches', 'Matches'), ('first', 'First matches'), ('rate', 'Match rate'), ('status', 'Signal')]),
)


## 8 · Optional Databricks appendix — production Spark boundary

Set `RUN_SPARK_APPENDIX = True` in Databricks to demonstrate the public lazy DataFrame API. This adds native typed results, detailed audit traces, explicit assignment application, and the production coverage aggregation. It does **not** publish metadata or create tables.


In [ ]:
RUN_SPARK_APPENDIX = False

if RUN_SPARK_APPENDIX:
    from rules_engine import RulesetCoverageAnalyzer, SparkRulesEngineRuntime

    input_df = spark.createDataFrame(source_rows)  # `spark` is supplied by Databricks.
    runtime = SparkRulesEngineRuntime(repository=None, function_registry=registry)
    evaluation = runtime.evaluate_dataframe(
        input_df,
        ruleset,
        key_columns=['account_id'],
        fail_on_error=False,
        full_audit=True,
    ).persist()
    try:
        display_value(evaluation.results_df.orderBy('account_id'))
        display_value(evaluation.apply_assignments().orderBy('account_id'))
        production_coverage = RulesetCoverageAnalyzer(runtime).analyze(
            input_df, ruleset, broad_match_threshold=0.40
        )
        display_value(production_coverage.rules)
        display_value(production_coverage.no_match_rows.orderBy('account_id'))
    finally:
        evaluation.unpersist()
else:
    render_html(
        "<div class='re-wrap re-callout'><strong>Portable demo mode</strong>"
        "<div class='re-note'>The Spark appendix is off. The core demo above remains fully executable without Java, a cluster, or metadata tables.</div></div>"
    )


## Close with these three points

1. **One contract:** authoring choices come from the engine that will execute them.
2. **Evidence by design:** matching, assignment state, version identity, and optional full audit are separate from the business payload.
3. **Safer operations:** strict compilation and coverage diagnostics move defects and rule gaps earlier in the lifecycle.

> Suggested final line: **“We can now change business logic faster without making the decision process harder to govern.”**
